In [ ]:
from __future__ import annotations

import json
import math
import re
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Iterable

try:
    import pandas as pd
except Exception as exc:
    raise RuntimeError("Install pandas to run this notebook.") from exc

try:
    import pdfplumber
except Exception:
    pdfplumber = None

try:
    import camelot
except Exception:
    camelot = None

try:
    import fitz
except Exception:
    fitz = None

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
DEFAULT_INPUT_DIR = DATA_DIR / "reports"
TEST_INPUT_DIR = DATA_DIR / "test"
OUTPUT_DIR = DATA_DIR / "processed_reports" / "tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"pdfplumber available: {pdfplumber is not None}")
print(f"camelot available: {camelot is not None}")
print(f"PyMuPDF available: {fitz is not None}")

## **Configuration**

In [ ]:
@dataclass
class TableExtractionConfig:
    input_dir: Path = TEST_INPUT_DIR
    output_dir: Path = OUTPUT_DIR
    max_pages: int | None = None
    min_rows: int = 2
    min_cols: int = 2
    caption_search_height: float = 90.0
    below_search_height: float = 45.0
    overlap_threshold: float = 0.72
    flatten_min_numeric_cells: int = 2
    flatten_min_numeric_ratio: float = 0.18
    write_json_sidecars: bool = True


CFG = TableExtractionConfig()
CFG

## **Data Contracts**

In [ ]:
@dataclass
class RawTableCandidate:
    source_file: str
    page_num: int
    table_idx: int
    extractor: str
    bbox: tuple[float, float, float, float] | None
    rows: list[list[Any]]
    caption_candidate: str | None = None
    context_above: str | None = None
    context_below: str | None = None
    quality_score: float | None = None
    quality_features: dict[str, Any] = field(default_factory=dict)


@dataclass
class NormalizedTable:
    source_file: str
    page_num: int
    table_idx: int
    extractor: str
    bbox: tuple[float, float, float, float] | None
    title: str | None
    context: str | None
    header_rows: list[list[str]]
    column_headers: list[str]
    stub_columns: list[str]
    body_rows: list[dict[str, Any]]
    footnotes: list[str]
    units: str | None
    currency: str | None
    confidence: float
    quality_features: dict[str, Any] = field(default_factory=dict)


def dataclass_records(items: Iterable[Any]) -> list[dict[str, Any]]:
    return [asdict(item) for item in items]


def to_jsonable(value: Any) -> Any:
    if isinstance(value, float) and math.isnan(value):
        return None
    if isinstance(value, Path):
        return str(value)
    return value

## **Text Utilities**

In [ ]:
def clean_text(value: Any) -> str:
    if value is None:
        return ""
    text = str(value).replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def normalize_cell(value: Any) -> str:
    return re.sub(r"\s+", " ", clean_text(value)).strip()


def normalize_header(value: Any) -> str:
    text = normalize_cell(value).lower()
    text = re.sub(r"[^0-9a-zA-Z%/$()., -]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text or "unnamed"


def is_blank(value: Any) -> bool:
    return normalize_cell(value) == ""


def parse_number(value: Any) -> float | None:
    text = normalize_cell(value)
    if not text or text.lower() in {"nan", "none", "null", "-", "--"}:
        return None

    negative = text.startswith("(") and text.endswith(")")
    pct = text.endswith("%")
    text = text.strip("()% ")
    text = text.replace(",", "").replace("−", "-")
    text = re.sub(r"[^0-9.\-]", "", text)
    if not text or text in {"-", "."}:
        return None
    try:
        value_float = float(text)
    except ValueError:
        return None
    if negative:
        value_float *= -1
    return value_float / 100.0 if pct else value_float


def looks_like_date_or_period(value: Any) -> bool:
    text = normalize_cell(value).lower()
    patterns = [
        r"\b20\d{2}\b",
        r"\b19\d{2}\b",
        r"\bq[1-4]\b",
        r"\bfy\s?\d{2,4}\b",
        r"\bjan(?:uary)?\b|\bfeb(?:ruary)?\b|\bmar(?:ch)?\b|\bapr(?:il)?\b|\bmay\b|\bjun(?:e)?\b|\bjul(?:y)?\b|\baug(?:ust)?\b|\bsep(?:tember)?\b|\boct(?:ober)?\b|\bnov(?:ember)?\b|\bdec(?:ember)?\b",
    ]
    return any(re.search(pattern, text, flags=re.I) for pattern in patterns)


def infer_units_and_currency(text: str) -> tuple[str | None, str | None]:
    lower = text.lower()
    currency = None
    if "$" in text or re.search(r"\busd\b|u\.s\. dollars", lower):
        currency = "USD"
    elif "€" in text or re.search(r"\beur\b", lower):
        currency = "EUR"
    elif "£" in text or re.search(r"\bgbp\b", lower):
        currency = "GBP"

    units = None
    if re.search(r"\bin millions\b|\bmillions\b|\$m\b", lower):
        units = "millions"
    elif re.search(r"\bin thousands\b|\bthousands\b|\$k\b", lower):
        units = "thousands"
    elif re.search(r"\bin billions\b|\bbillions\b|\$b\b", lower):
        units = "billions"
    elif "%" in text or re.search(r"\bpercent(?:age)?\b", lower):
        units = "percent"
    return units, currency

## **PDF Context**

In [ ]:
def crop_text(page: Any, bbox: tuple[float, float, float, float] | None) -> str:
    if bbox is None:
        return ""
    try:
        return clean_text(page.crop(bbox).extract_text() or "")
    except Exception:
        return ""


def page_context_for_bbox(page: Any, bbox: tuple[float, float, float, float] | None, cfg: TableExtractionConfig) -> tuple[str | None, str | None, str | None]:
    if bbox is None:
        text = clean_text(page.extract_text() or "")
        lines = [line.strip() for line in text.splitlines() if line.strip()]
        caption = lines[0] if lines else None
        return caption, None, None

    x0, top, x1, bottom = bbox
    page_width = float(getattr(page, "width", x1))
    page_height = float(getattr(page, "height", bottom))

    above_bbox = (0, max(0, top - cfg.caption_search_height), page_width, max(0, top - 2))
    below_bbox = (0, min(page_height, bottom + 2), page_width, min(page_height, bottom + cfg.below_search_height))
    above = crop_text(page, above_bbox)
    below = crop_text(page, below_bbox)

    lines = [line.strip() for line in above.splitlines() if line.strip()]
    caption = lines[-1] if lines else None
    return caption, above or None, below or None

## **Candidate Extraction**

In [ ]:
PDFPLUMBER_STRATEGIES = [
    {
        "name": "pdfplumber_lines",
        "settings": {
            "vertical_strategy": "lines",
            "horizontal_strategy": "lines",
            "intersection_tolerance": 5,
            "snap_tolerance": 3,
            "join_tolerance": 3,
        },
    },
    {
        "name": "pdfplumber_text",
        "settings": {
            "vertical_strategy": "text",
            "horizontal_strategy": "text",
            "text_tolerance": 3,
            "snap_tolerance": 3,
            "join_tolerance": 3,
        },
    },
]


def extract_pdfplumber_candidates(pdf_path: Path, cfg: TableExtractionConfig) -> list[RawTableCandidate]:
    if pdfplumber is None:
        return []

    candidates: list[RawTableCandidate] = []
    with pdfplumber.open(str(pdf_path)) as pdf:
        pages = pdf.pages[: cfg.max_pages] if cfg.max_pages else pdf.pages
        for page_idx, page in enumerate(pages, start=1):
            for strategy in PDFPLUMBER_STRATEGIES:
                try:
                    tables = page.find_tables(table_settings=strategy["settings"])
                except Exception:
                    tables = []

                for table_idx, table in enumerate(tables):
                    try:
                        rows = table.extract() or []
                    except Exception:
                        rows = []
                    if not rows:
                        continue
                    bbox = tuple(float(x) for x in table.bbox) if getattr(table, "bbox", None) else None
                    caption, above, below = page_context_for_bbox(page, bbox, cfg)
                    candidates.append(RawTableCandidate(
                        source_file=str(pdf_path),
                        page_num=page_idx,
                        table_idx=table_idx,
                        extractor=strategy["name"],
                        bbox=bbox,
                        rows=rows,
                        caption_candidate=caption,
                        context_above=above,
                        context_below=below,
                    ))
    return candidates


def extract_camelot_candidates(pdf_path: Path, cfg: TableExtractionConfig) -> list[RawTableCandidate]:
    if camelot is None:
        return []

    candidates: list[RawTableCandidate] = []
    page_spec = f"1-{cfg.max_pages}" if cfg.max_pages else "all"
    for flavor in ["lattice", "stream"]:
        try:
            tables = camelot.read_pdf(str(pdf_path), pages=page_spec, flavor=flavor)
        except Exception:
            tables = []
        for idx, table in enumerate(tables):
            df = table.df.copy()
            rows = df.where(df.notna(), None).values.tolist()
            page_num = int(getattr(table, "page", 0) or 0)
            candidates.append(RawTableCandidate(
                source_file=str(pdf_path),
                page_num=page_num,
                table_idx=idx,
                extractor=f"camelot_{flavor}",
                bbox=None,
                rows=rows,
                quality_features={"camelot_accuracy": getattr(table, "accuracy", None)},
            ))
    return candidates


def extract_raw_table_candidates(pdf_path: Path, cfg: TableExtractionConfig = CFG) -> list[RawTableCandidate]:
    candidates = []
    candidates.extend(extract_pdfplumber_candidates(pdf_path, cfg))
    candidates.extend(extract_camelot_candidates(pdf_path, cfg))
    return candidates

## **Quality Scoring**

In [ ]:
def table_dimensions(rows: list[list[Any]]) -> tuple[int, int]:
    n_rows = len(rows)
    n_cols = max((len(row) for row in rows), default=0)
    return n_rows, n_cols


def rectangularize(rows: list[list[Any]]) -> list[list[Any]]:
    n_cols = table_dimensions(rows)[1]
    return [list(row) + [None] * (n_cols - len(row)) for row in rows]


def score_table_candidate(candidate: RawTableCandidate, cfg: TableExtractionConfig = CFG) -> RawTableCandidate:
    rows = rectangularize(candidate.rows)
    n_rows, n_cols = table_dimensions(rows)
    cells = [cell for row in rows for cell in row]
    total_cells = max(len(cells), 1)

    non_empty = sum(not is_blank(cell) for cell in cells)
    numeric = sum(parse_number(cell) is not None for cell in cells)
    date_like = sum(looks_like_date_or_period(cell) for cell in cells)
    row_widths = [sum(not is_blank(cell) for cell in row) for row in rows]
    width_consistency = 1.0 - (pd.Series(row_widths).std(ddof=0) / max(n_cols, 1) if row_widths else 1.0)
    width_consistency = max(0.0, min(float(width_consistency), 1.0))

    fill_ratio = non_empty / total_cells
    numeric_ratio = numeric / total_cells
    date_ratio = date_like / total_cells
    shape_score = min(n_rows / 8, 1.0) * 0.45 + min(n_cols / 5, 1.0) * 0.55
    extractor_bonus = 0.08 if candidate.extractor.endswith("lines") or "lattice" in candidate.extractor else 0.03

    score = (
        0.28 * fill_ratio
        + 0.22 * width_consistency
        + 0.18 * shape_score
        + 0.17 * min(numeric_ratio * 2.0, 1.0)
        + 0.07 * min(date_ratio * 3.0, 1.0)
        + extractor_bonus
    )
    if n_rows < cfg.min_rows or n_cols < cfg.min_cols:
        score *= 0.25

    features = {
        **candidate.quality_features,
        "n_rows": n_rows,
        "n_cols": n_cols,
        "fill_ratio": round(fill_ratio, 4),
        "numeric_ratio": round(numeric_ratio, 4),
        "date_ratio": round(date_ratio, 4),
        "width_consistency": round(width_consistency, 4),
    }
    candidate.quality_score = round(float(score), 4)
    candidate.quality_features = features
    return candidate


def bbox_area(bbox: tuple[float, float, float, float] | None) -> float:
    if bbox is None:
        return 0.0
    x0, y0, x1, y1 = bbox
    return max(0.0, x1 - x0) * max(0.0, y1 - y0)


def bbox_iou(a: tuple[float, float, float, float] | None, b: tuple[float, float, float, float] | None) -> float:
    if a is None or b is None:
        return 0.0
    ax0, ay0, ax1, ay1 = a
    bx0, by0, bx1, by1 = b
    ix0, iy0 = max(ax0, bx0), max(ay0, by0)
    ix1, iy1 = min(ax1, bx1), min(ay1, by1)
    intersection = bbox_area((ix0, iy0, ix1, iy1))
    union = bbox_area(a) + bbox_area(b) - intersection
    return intersection / union if union else 0.0


def deduplicate_candidates(candidates: list[RawTableCandidate], cfg: TableExtractionConfig = CFG) -> list[RawTableCandidate]:
    scored = sorted((score_table_candidate(c, cfg) for c in candidates), key=lambda c: c.quality_score or 0, reverse=True)
    kept: list[RawTableCandidate] = []
    for candidate in scored:
        overlaps_existing = any(
            candidate.source_file == other.source_file
            and candidate.page_num == other.page_num
            and bbox_iou(candidate.bbox, other.bbox) >= cfg.overlap_threshold
            for other in kept
        )
        if not overlaps_existing:
            kept.append(candidate)
    return sorted(kept, key=lambda c: (c.source_file, c.page_num, c.table_idx, c.extractor))

## **Generic Normalization**

In [ ]:
def infer_header_row_count(rows: list[list[Any]], max_header_rows: int = 4) -> int:
    rows = rectangularize(rows)
    if len(rows) <= 1:
        return 0

    candidates = rows[: min(max_header_rows, len(rows) - 1)]
    header_count = 0
    for row in candidates:
        non_blank = [cell for cell in row if not is_blank(cell)]
        if not non_blank:
            header_count += 1
            continue
        numeric_ratio = sum(parse_number(cell) is not None for cell in non_blank) / len(non_blank)
        date_ratio = sum(looks_like_date_or_period(cell) for cell in non_blank) / len(non_blank)
        text_ratio = sum(parse_number(cell) is None for cell in non_blank) / len(non_blank)
        if numeric_ratio <= 0.35 and (text_ratio >= 0.55 or date_ratio >= 0.25):
            header_count += 1
        else:
            break
    return min(max(header_count, 1), len(rows) - 1)


def combine_column_headers(header_rows: list[list[Any]], n_cols: int) -> list[str]:
    if not header_rows:
        return [f"col_{idx}" for idx in range(n_cols)]

    headers = []
    previous_by_row = [""] * len(header_rows)
    for col_idx in range(n_cols):
        parts = []
        for row_idx, row in enumerate(header_rows):
            raw = row[col_idx] if col_idx < len(row) else None
            text = normalize_cell(raw)
            if text:
                previous_by_row[row_idx] = text
                parts.append(text)
            elif previous_by_row[row_idx] and col_idx > 0:
                parts.append(previous_by_row[row_idx])
        header = " | ".join(dict.fromkeys(part for part in parts if part))
        headers.append(normalize_header(header or f"col_{col_idx}"))

    seen: dict[str, int] = {}
    unique_headers = []
    for header in headers:
        count = seen.get(header, 0)
        seen[header] = count + 1
        unique_headers.append(header if count == 0 else f"{header}_{count + 1}")
    return unique_headers


def infer_stub_columns(body_rows: list[list[Any]], headers: list[str]) -> list[str]:
    stub_headers = []
    n_cols = len(headers)
    for col_idx, header in enumerate(headers):
        values = [row[col_idx] for row in body_rows if col_idx < len(row)]
        non_blank = [value for value in values if not is_blank(value)]
        if not non_blank:
            continue
        numeric_ratio = sum(parse_number(value) is not None for value in non_blank) / len(non_blank)
        date_ratio = sum(looks_like_date_or_period(value) for value in non_blank) / len(non_blank)
        if col_idx == 0 and numeric_ratio < 0.35:
            stub_headers.append(header)
        elif col_idx < min(2, n_cols) and numeric_ratio < 0.20 and date_ratio < 0.20:
            stub_headers.append(header)
    return stub_headers


def split_body_and_footnotes(body_rows: list[list[Any]], headers: list[str]) -> tuple[list[list[Any]], list[str]]:
    data_rows = []
    footnotes = []
    for row in body_rows:
        non_blank = [normalize_cell(cell) for cell in row if not is_blank(cell)]
        if len(non_blank) == 1 and len(row) > 2:
            text = non_blank[0]
            if re.search(r"^(note|\(\d\)|\*|source:|includes?)", text, flags=re.I) or len(text) > 60:
                footnotes.append(text)
                continue
        data_rows.append(row)
    return data_rows, footnotes


def normalize_table(candidate: RawTableCandidate) -> NormalizedTable:
    rows = rectangularize(candidate.rows)
    n_rows, n_cols = table_dimensions(rows)
    header_count = infer_header_row_count(rows)
    header_rows_raw = rows[:header_count]
    body_rows_raw = rows[header_count:]
    headers = combine_column_headers(header_rows_raw, n_cols)
    body_rows_raw, footnotes = split_body_and_footnotes(body_rows_raw, headers)
    stub_columns = infer_stub_columns(body_rows_raw, headers)

    body_rows = []
    for row_idx, row in enumerate(body_rows_raw):
        row_dict = {headers[col_idx]: normalize_cell(row[col_idx]) for col_idx in range(n_cols)}
        row_dict["_row_idx"] = row_idx
        body_rows.append(row_dict)

    context = "\n".join(part for part in [candidate.context_above, candidate.context_below] if part)
    units, currency = infer_units_and_currency("\n".join([
        candidate.caption_candidate or "",
        context,
        " ".join(" ".join(map(str, row)) for row in rows[: min(3, len(rows))]),
    ]))

    confidence = float(candidate.quality_score or 0.0)
    if headers and body_rows:
        confidence = min(1.0, confidence + 0.08)
    if stub_columns:
        confidence = min(1.0, confidence + 0.04)

    return NormalizedTable(
        source_file=candidate.source_file,
        page_num=candidate.page_num,
        table_idx=candidate.table_idx,
        extractor=candidate.extractor,
        bbox=candidate.bbox,
        title=candidate.caption_candidate,
        context=context or None,
        header_rows=[[normalize_cell(cell) for cell in row] for row in header_rows_raw],
        column_headers=headers,
        stub_columns=stub_columns,
        body_rows=body_rows,
        footnotes=footnotes,
        units=units,
        currency=currency,
        confidence=round(confidence, 4),
        quality_features=candidate.quality_features,
    )

## **Metric Flattening**

In [ ]:
def row_label_for_record(row: dict[str, Any], stub_columns: list[str]) -> str | None:
    parts = [normalize_cell(row.get(col)) for col in stub_columns if normalize_cell(row.get(col))]
    if parts:
        return " | ".join(parts)
    for key, value in row.items():
        if key.startswith("_"):
            continue
        text = normalize_cell(value)
        if text and parse_number(text) is None:
            return text
    return None


def should_flatten_table(table: NormalizedTable, cfg: TableExtractionConfig = CFG) -> bool:
    values = []
    for row in table.body_rows:
        for col, value in row.items():
            if col.startswith("_") or col in table.stub_columns:
                continue
            values.append(value)
    if not values:
        return False
    numeric_count = sum(parse_number(value) is not None for value in values)
    numeric_ratio = numeric_count / len(values)
    return numeric_count >= cfg.flatten_min_numeric_cells and numeric_ratio >= cfg.flatten_min_numeric_ratio


def infer_period_from_header(header: str) -> dict[str, Any]:
    text = normalize_cell(header)
    year_match = re.search(r"\b(20\d{2}|19\d{2})\b", text)
    quarter_match = re.search(r"\bQ([1-4])\b", text, flags=re.I)
    return {
        "period_label": text or None,
        "period_year": int(year_match.group(1)) if year_match else None,
        "period_quarter": f"Q{quarter_match.group(1)}" if quarter_match else None,
    }


def normalized_table_to_metric_rows(table: NormalizedTable, cfg: TableExtractionConfig = CFG) -> list[dict[str, Any]]:
    if not should_flatten_table(table, cfg):
        return []

    rows = []
    for body_row in table.body_rows:
        row_label = row_label_for_record(body_row, table.stub_columns)
        for column, raw_value in body_row.items():
            if column.startswith("_") or column in table.stub_columns:
                continue
            value = parse_number(raw_value)
            if value is None:
                continue
            period_fields = infer_period_from_header(column)
            rows.append({
                "source_file": table.source_file,
                "source_type": "pdf_table_generic",
                "page_num": table.page_num,
                "table_idx": table.table_idx,
                "extractor": table.extractor,
                "table_title": table.title,
                "table_context": table.context,
                "row_idx": body_row.get("_row_idx"),
                "row_label": row_label,
                "metric": column,
                "value": value,
                "raw_value": raw_value,
                "units": table.units,
                "currency": table.currency,
                "confidence": table.confidence,
                **period_fields,
            })
    return rows

## **Persistence**

In [ ]:
def write_dataframe(df: pd.DataFrame, parquet_path: Path) -> Path:
    parquet_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_parquet(parquet_path, index=False)
        return parquet_path
    except Exception as exc:
        csv_path = parquet_path.with_suffix(".csv")
        df.to_csv(csv_path, index=False)
        print(f"Could not write parquet ({type(exc).__name__}: {exc}); wrote CSV instead: {csv_path}")
        return csv_path


def write_json_records(records: list[dict[str, Any]], path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(records, default=to_jsonable, indent=2), encoding="utf-8")
    return path


def normalized_tables_frame(tables: list[NormalizedTable]) -> pd.DataFrame:
    records = []
    for table in tables:
        rec = asdict(table)
        for key in ["bbox", "header_rows", "column_headers", "stub_columns", "body_rows", "footnotes", "quality_features"]:
            rec[key] = json.dumps(rec[key], default=to_jsonable)
        records.append(rec)
    return pd.DataFrame(records)


def raw_candidates_frame(candidates: list[RawTableCandidate]) -> pd.DataFrame:
    records = []
    for candidate in candidates:
        rec = asdict(candidate)
        for key in ["bbox", "rows", "quality_features"]:
            rec[key] = json.dumps(rec[key], default=to_jsonable)
        records.append(rec)
    return pd.DataFrame(records)

## **Pipeline**

In [ ]:
def discover_pdfs(root: Path) -> list[Path]:
    return sorted(path for path in root.rglob("*.pdf") if path.is_file())


def ingest_pdf_tables_generic(pdf_path: Path, cfg: TableExtractionConfig = CFG) -> dict[str, Any]:
    raw_candidates = extract_raw_table_candidates(pdf_path, cfg)
    selected_candidates = deduplicate_candidates(raw_candidates, cfg)
    normalized_tables = [normalize_table(candidate) for candidate in selected_candidates]
    metric_rows = [row for table in normalized_tables for row in normalized_table_to_metric_rows(table, cfg)]

    return {
        "source_file": str(pdf_path),
        "raw_candidates": raw_candidates,
        "selected_candidates": selected_candidates,
        "normalized_tables": normalized_tables,
        "metric_rows": metric_rows,
    }


def ingest_pdf_directory_generic(root: Path, cfg: TableExtractionConfig = CFG) -> dict[str, pd.DataFrame]:
    all_raw: list[RawTableCandidate] = []
    all_selected: list[RawTableCandidate] = []
    all_normalized: list[NormalizedTable] = []
    all_metrics: list[dict[str, Any]] = []

    for pdf_path in discover_pdfs(root):
        print(f"Extracting tables: {pdf_path}")
        result = ingest_pdf_tables_generic(pdf_path, cfg)
        all_raw.extend(result["raw_candidates"])
        all_selected.extend(result["selected_candidates"])
        all_normalized.extend(result["normalized_tables"])
        all_metrics.extend(result["metric_rows"])

    raw_df = raw_candidates_frame(all_raw)
    selected_df = raw_candidates_frame(all_selected)
    normalized_df = normalized_tables_frame(all_normalized)
    metrics_df = pd.DataFrame(all_metrics)

    paths = {
        "raw_pdf_table_candidates": write_dataframe(raw_df, cfg.output_dir / "raw_pdf_table_candidates.parquet"),
        "selected_pdf_tables": write_dataframe(selected_df, cfg.output_dir / "selected_pdf_tables.parquet"),
        "normalized_pdf_tables": write_dataframe(normalized_df, cfg.output_dir / "normalized_pdf_tables.parquet"),
        "generic_pdf_metrics": write_dataframe(metrics_df, cfg.output_dir / "generic_pdf_metrics.parquet"),
    }

    if cfg.write_json_sidecars:
        paths["normalized_pdf_tables_json"] = write_json_records(dataclass_records(all_normalized), cfg.output_dir / "normalized_pdf_tables.json")

    return {
        "raw_candidates": raw_df,
        "selected_tables": selected_df,
        "normalized_tables": normalized_df,
        "metrics": metrics_df,
        "paths": paths,
    }

## **Inspection**

In [ ]:
sample_pdf = TEST_INPUT_DIR / "MSFT_FY25Q1_10Q.pdf"
if sample_pdf.exists():
    sample_result = ingest_pdf_tables_generic(sample_pdf, CFG)
    print("raw candidates:", len(sample_result["raw_candidates"]))
    print("selected tables:", len(sample_result["selected_candidates"]))
    print("normalized tables:", len(sample_result["normalized_tables"]))
    print("metric rows:", len(sample_result["metric_rows"]))
else:
    print(f"Sample PDF not found: {sample_pdf}")

In [ ]:
if sample_pdf.exists() and sample_result["normalized_tables"]:
    preview_table = sample_result["normalized_tables"][0]
    print("title:", preview_table.title)
    print("headers:", preview_table.column_headers)
    print("stub columns:", preview_table.stub_columns)
    display(pd.DataFrame(preview_table.body_rows).head(10))

if sample_pdf.exists() and sample_result["metric_rows"]:
    display(pd.DataFrame(sample_result["metric_rows"]).head(20))

## **Batch Run**

In [ ]:
# outputs = ingest_pdf_directory_generic(CFG.input_dir, CFG)
# outputs["paths"]

## **Quality Dashboard**

In [ ]:
def quality_summary(selected_tables: pd.DataFrame, metrics: pd.DataFrame) -> pd.DataFrame:
    if selected_tables.empty:
        return pd.DataFrame()
    summary = selected_tables.copy()
    if "quality_features" in summary.columns:
        features = summary["quality_features"].map(lambda x: json.loads(x) if isinstance(x, str) and x else {})
        for key in ["n_rows", "n_cols", "fill_ratio", "numeric_ratio", "width_consistency"]:
            summary[key] = features.map(lambda item: item.get(key))
    metric_counts = pd.DataFrame()
    if not metrics.empty and {"source_file", "page_num", "table_idx"}.issubset(metrics.columns):
        metric_counts = metrics.groupby(["source_file", "page_num", "table_idx"]).size().rename("metric_rows").reset_index()
        summary = summary.merge(metric_counts, on=["source_file", "page_num", "table_idx"], how="left")
    summary["metric_rows"] = summary.get("metric_rows", 0).fillna(0).astype(int)
    columns = [
        "source_file", "page_num", "table_idx", "extractor", "quality_score",
        "n_rows", "n_cols", "fill_ratio", "numeric_ratio", "width_consistency", "metric_rows",
    ]
    return summary[[col for col in columns if col in summary.columns]].sort_values(["source_file", "page_num", "table_idx"])


if sample_pdf.exists():
    sample_selected_df = raw_candidates_frame(sample_result["selected_candidates"])
    sample_metrics_df = pd.DataFrame(sample_result["metric_rows"])
    display(quality_summary(sample_selected_df, sample_metrics_df).head(30))